# 🔗 Module 5: Correlation — Real-Life Cases

**The Story:** PizzaChain wants to understand which factors drive sales.
- Does spending more on ads lead to more sales?
- Do stores with more employees deliver faster?
- Does customer rating predict revenue?

Correlation helps us find these relationships — but with a critical warning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
n = 50

stores = pd.DataFrame({
    'store_id': [f'Store_{i+1}' for i in range(n)],
    'daily_sales': np.random.normal(500, 100, n),
    'ad_spend': np.random.normal(200, 50, n),
    'num_employees': np.random.randint(8, 22, n),
    'customer_rating': np.round(np.random.uniform(3.0, 5.0, n), 1),
    'delivery_time_min': np.random.normal(30, 7, n),
})

# Create realistic correlations
stores['daily_sales'] = 200 + 1.5 * stores['ad_spend'] + np.random.normal(0, 60, n)
stores['delivery_time_min'] = 50 - 1.0 * stores['num_employees'] + np.random.normal(0, 5, n)
# Rating is mostly random (weak correlation)
stores['customer_rating'] = np.round(3.0 + 0.001 * stores['daily_sales'] + np.random.normal(0, 0.5, n), 1).clip(1, 5)

stores = stores.round(1)
print(f'Dataset: {len(stores)} stores')
stores.head(10)

---
## Case 1: Strong Positive Correlation — Ad Spend vs Sales

**Question:** Does spending more on ads lead to more sales?

**Pearson correlation (r):** Measures LINEAR relationship, -1 to +1
```
r = Cov(X,Y) / (σ_X × σ_Y)
```

In [ ]:
x = stores['ad_spend']
y = stores['daily_sales']

# Step-by-step
cov_xy = np.cov(x, y)[0, 1]
std_x = x.std()
std_y = y.std()
r_manual = cov_xy / (std_x * std_y)
r_scipy, p_corr = stats.pearsonr(x, y)

print('📊 CORRELATION: STEP-BY-STEP MATH')
print('=' * 55)
print(f'  Cov(ad_spend, sales) = {cov_xy:.1f}')
print(f'  σ_ad_spend           = {std_x:.1f}')
print(f'  σ_sales              = {std_y:.1f}')
print(f'  r = Cov / (σ_x × σ_y) = {cov_xy:.1f} / ({std_x:.1f} × {std_y:.1f}) = {r_manual:.3f}')
print(f'  (scipy confirms: r = {r_scipy:.3f}, p = {p_corr:.6f})')
print()
strength = 'Very strong' if abs(r_scipy) > 0.8 else 'Strong' if abs(r_scipy) > 0.6 else 'Moderate' if abs(r_scipy) > 0.4 else 'Weak'
print(f'💡 INFERENCE:')
print(f'   r = {r_scipy:.2f} → {strength} positive correlation.')
print(f'   Stores that spend more on ads tend to have higher sales.')
print(f'   p = {p_corr:.6f} → This relationship is statistically significant.')
print(f'   ⚠️ BUT: Does ad spend CAUSE more sales, or do profitable stores just spend more?')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter with trend line
axes[0].scatter(x, y, color='#7c6aff', alpha=0.6, s=50, edgecolors='white')
z = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
axes[0].plot(x_line, np.polyval(z, x_line), color='#f45d6d', linewidth=2, linestyle='--', label=f'Trend (r={r_scipy:.2f})')
axes[0].set_xlabel('Ad Spend ($)')
axes[0].set_ylabel('Daily Sales ($)')
axes[0].set_title(f'Ad Spend vs Sales (r = {r_scipy:.2f})', fontsize=12, fontweight='bold')
axes[0].legend()

# Full heatmap
numeric = stores[['daily_sales', 'ad_spend', 'num_employees', 'customer_rating', 'delivery_time_min']]
corr = numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=sns.diverging_palette(10, 150, as_cmap=True),
            center=0, vmin=-1, vmax=1, square=True, linewidths=1, ax=axes[1],
            annot_kws={'size': 11, 'fontweight': 'bold'})
axes[1].set_title('Full Correlation Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Case 2: Negative Correlation — Employees vs Delivery Time

**Question:** Do stores with more employees deliver faster?

In [ ]:
x2 = stores['num_employees']
y2 = stores['delivery_time_min']
r2, p2 = stats.pearsonr(x2, y2)

print(f'📊 r = {r2:.2f}, p = {p2:.4f}')
print(f'💡 INFERENCE: r = {r2:.2f} → Strong NEGATIVE correlation.')
print(f'   More employees → shorter delivery times.')
print(f'   Each additional employee reduces delivery by ~{abs(np.polyfit(x2, y2, 1)[0]):.1f} minutes.')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x2, y2, color='#22d3a7', alpha=0.6, s=50, edgecolors='white')
z2 = np.polyfit(x2, y2, 1)
ax.plot(np.linspace(x2.min(), x2.max(), 100), np.polyval(z2, np.linspace(x2.min(), x2.max(), 100)),
        color='#f45d6d', linewidth=2, linestyle='--')
ax.set_xlabel('Number of Employees')
ax.set_ylabel('Delivery Time (min)')
ax.set_title(f'More Staff → Faster Delivery (r = {r2:.2f})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Case 3: No Correlation — Rating vs Sales

**Question:** Do higher-rated stores sell more?

In [ ]:
x3 = stores['customer_rating']
y3 = stores['daily_sales']
r3, p3 = stats.pearsonr(x3, y3)

print(f'📊 r = {r3:.2f}, p = {p3:.4f}')
print(f'💡 INFERENCE: r = {r3:.2f} → Weak/no correlation.')
print(f'   Customer rating does NOT predict sales in our data.')
print(f'   p = {p3:.4f} {"< 0.05 (but effect is tiny)" if p3 < 0.05 else "> 0.05 (not even significant)"}')
print(f'   ACTION: Don\'t use rating as a sales predictor. Look elsewhere.')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x3, y3, color='#f5b731', alpha=0.6, s=50, edgecolors='white')
ax.set_xlabel('Customer Rating')
ax.set_ylabel('Daily Sales ($)')
ax.set_title(f'Rating vs Sales — No Pattern (r = {r3:.2f})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚠️ The Causation Trap

We found: ad_spend ↔ sales (r ≈ 0.9). Does that mean ads CAUSE sales?

**Not necessarily!** Possible explanations:
1. Ads → More customers → More sales (ads cause sales) ✅
2. More sales → More budget → More ad spend (sales cause ads) 🔄
3. Good location → Both more sales AND more ad budget (confounding variable) ⚠️

**Only a controlled experiment (A/B test) can prove causation.**

---
## 📝 Summary

| Pair | r | Strength | Significant? | Business Insight |
|---|---|---|---|---|
| Ad spend ↔ Sales | ~0.9 | Very strong + | Yes | More ads → more sales (but verify causation) |
| Employees ↔ Delivery | ~-0.8 | Strong − | Yes | More staff → faster delivery |
| Rating ↔ Sales | ~0.1 | None | No | Rating doesn't predict sales |